# Hong Kong Weather ETL Pipeline

使用 Open-Meteo API 拉取香港多个地点的小时天气数据，存入 DuckDB，并生成日汇总表。

## 项目目标
- 从 Open-Meteo API 抽取数据（Extract）
- 清洗、合并、计算日均温度 / 总降水量等衍生字段（Transform）
- 写入 DuckDB 的 `raw_weather` 和 `daily_weather_summary` 两张表（Load）
- 支持防重复写入，可重复运行

## 技术栈
- Python
- requests + pandas
- DuckDB
- Google Colab

## 如何运行
1. 修改脚本中的 `locations` 列表或 `FORECAST_DAYS`
2. 运行完整脚本单元格即可

## 数据表说明
- `raw_weather`：原始小时数据（地点 + 时间 + 温度/湿度/降水/风速/云量）
- `daily_weather_summary`：按地点+日期聚合的日汇总数据

## 查询示例
脚本下方附有简单的查询单元格，可用于：
- 查看已存储的日期范围和地点数量
- 检查历史数据是否正常保留

In [ ]:
import requests
import pandas as pd
import duckdb
from datetime import datetime

# ==================== 可修改的配置 ====================
# 1. 地点列表（想加/减地点，直接改这里）
locations = [
    {"name": "Central", "lat": 22.282, "lon": 114.158},
    {"name": "Tsim Sha Tsui", "lat": 22.299, "lon": 114.172},
    {"name": "Sha Tin", "lat": 22.383, "lon": 114.188},
    {"name": "HK Airport", "lat": 22.309, "lon": 113.914},
    {"name": "Tuen Mun", "lat": 22.392, "lon": 113.977},
    {"name": "Tsing Yi", "lat": 22.345, "lon": 114.101},
    {"name": "Sai Kung", "lat": 22.383, "lon": 114.267},
    {"name": "Tseung Kwan O", "lat": 22.328, "lon": 114.250},
    {"name": "Sheung Shui", "lat": 22.510, "lon": 114.127},
]

# 2. 预报天数（想看更多天，改这个数字）
FORECAST_DAYS = 2

# 3. 数据库文件名
DB_FILE = "weather.duckdb"
# =====================================================

def fetch_hourly(loc, days):
    """拉取单个地点的小时数据"""
    url = "https://api.open-meteo.com/v1/forecast"
    params = {
        "latitude": loc["lat"],
        "longitude": loc["lon"],
        "hourly": "temperature_2m,relative_humidity_2m,precipitation,wind_speed_10m,cloud_cover",
        "timezone": "Asia/Hong_Kong",
        "forecast_days": days,
    }
    response = requests.get(url, params=params)
    response.raise_for_status()
    data = response.json()

    df = pd.DataFrame(data["hourly"])
    df["location"] = loc["name"]
    return df

# ---------- 1. 拉取所有地点数据 ----------
print("开始拉取数据...")
all_dfs = []
for loc in locations:
    print(f"  正在拉取：{loc['name']}")
    df_loc = fetch_hourly(loc, FORECAST_DAYS)
    all_dfs.append(df_loc)

raw_df = pd.concat(all_dfs, ignore_index=True)
raw_df["extracted_at"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
print(f"共拉取 {len(raw_df)} 条小时数据\n")

# ---------- 2. 连接数据库并写入 raw_weather（防重复） ----------
con = duckdb.connect(DB_FILE)

# 如果表不存在则创建
con.execute("""
    CREATE TABLE IF NOT EXISTS raw_weather (
        time VARCHAR,
        temperature_2m DOUBLE,
        relative_humidity_2m BIGINT,
        precipitation DOUBLE,
        wind_speed_10m DOUBLE,
        cloud_cover BIGINT,
        location VARCHAR,
        extracted_at VARCHAR
    )
""")

# 防重复写入
con.execute("CREATE OR REPLACE TEMP TABLE new_raw AS SELECT * FROM raw_df")
con.execute("""
    DELETE FROM raw_weather
    WHERE (location, time) IN (SELECT location, time FROM new_raw)
""")
con.execute("INSERT INTO raw_weather SELECT * FROM new_raw")

print("raw_weather 写入完成")
print(con.execute("SELECT COUNT(*) AS total_rows FROM raw_weather").df())

# ---------- 3. 重新生成日汇总表 ----------
con.execute("""
    CREATE OR REPLACE TABLE daily_weather_summary AS
    SELECT
        location,
        CAST(time AS DATE) AS date,
        ROUND(AVG(temperature_2m), 1) AS avg_temp,
        ROUND(AVG(relative_humidity_2m), 1) AS avg_humidity,
        ROUND(SUM(precipitation), 1) AS total_precip,
        ROUND(AVG(wind_speed_10m), 1) AS avg_wind,
        ROUND(AVG(cloud_cover), 1) AS avg_cloud,
        MAX(extracted_at) AS extracted_at
    FROM raw_weather
    GROUP BY location, CAST(time AS DATE)
    ORDER BY location, date
""")

print("\ndaily_weather_summary 生成完成")
print(con.execute("SELECT COUNT(*) AS total_rows FROM daily_weather_summary").df())
print("\n前 5 行日汇总：")
print(con.execute("SELECT * FROM daily_weather_summary LIMIT 5").df())

con.close()
print("\n脚本执行完毕。")

开始拉取数据...
  正在拉取：Central
  正在拉取：Tsim Sha Tsui
  正在拉取：Sha Tin
  正在拉取：HK Airport
  正在拉取：Tuen Mun
  正在拉取：Tsing Yi
  正在拉取：Sai Kung
  正在拉取：Tseung Kwan O
  正在拉取：Sheung Shui
共拉取 432 条小时数据

raw_weather 写入完成
   total_rows
0         432

daily_weather_summary 生成完成
   total_rows
0          18

前 5 行日汇总：
     location       date  avg_temp  avg_humidity  total_precip  avg_wind  \
0     Central 2026-10-17      27.1          76.5           1.6      15.9   
1     Central 2026-10-18      27.4          77.8           0.6      13.3   
2  HK Airport 2026-10-17      27.2          77.8           0.2      16.0   
3  HK Airport 2026-10-18      27.5          78.5           0.4      13.7   
4    Sai Kung 2026-10-17      25.8          82.3           1.6      12.2   

   avg_cloud         extracted_at  
0       41.4  2026-10-17 14:44:52  
1       26.5  2026-10-17 14:44:52  
2       26.1  2026-10-17 14:44:52  
3       21.0  2026-10-17 14:44:52  
4       45.8  2026-10-17 14:44:52  

脚本执行完毕。


In [ ]:
import duckdb
con = duckdb.connect("weather.duckdb")

In [ ]:
print(con.execute("""
    SELECT date, COUNT(*) AS location_count
    FROM daily_weather_summary
    GROUP BY date
    ORDER BY date
""").df())

        date  location_count
0 2026-10-17               9
1 2026-10-18               9
